In [ ]:
# Import python packages
import streamlit as st
import pandas as pd

# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()


In [ ]:
-- Welcome to Snowflake Notebooks!
-- Try out a SQL cell to generate some data.
SELECT 'FRIDAY' as SNOWDAY, 0.2 as CHANCE_OF_SNOW
UNION ALL
SELECT 'SATURDAY',0.5
UNION ALL 
SELECT 'SUNDAY', 0.9;

In [ ]:
# Then, we can use the python name to turn cell2 into a Pandas dataframe
my_df = cell2.to_pandas()

# Chart the data
st.subheader("Chance of SNOW ❄️")
st.line_chart(my_df, x='SNOWDAY', y='CHANCE_OF_SNOW')

# Give it a go!
st.subheader("Try it out yourself and show off your skills 🥇")

In [ ]:
CREATE OR REPLACE WAREHOUSE RETAIL_WH
  WAREHOUSE_SIZE = XSMALL
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE;


In [ ]:
CREATE OR REPLACE DATABASE RETAIL_DW;

-- Schemas
CREATE OR REPLACE SCHEMA RAW;         -- landing zone for CSVs
CREATE OR REPLACE SCHEMA DIM;         -- dimensions
CREATE OR REPLACE SCHEMA FACT;        -- facts
CREATE OR REPLACE SCHEMA TRANSFORM;   -- procs, streams, tasks
CREATE OR REPLACE SCHEMA ANALYTICS;   -- final views


In [ ]:
USE DATABASE RETAIL_DW;
USE SCHEMA RAW;

CREATE OR REPLACE STAGE RETAIL_STAGE
  FILE_FORMAT = (TYPE = CSV FIELD_OPTIONALLY_ENCLOSED_BY = '"' SKIP_HEADER = 1);


In [ ]:
-- customers (11 columns)
CREATE OR REPLACE TABLE RAW.CUSTOMERS_RAW (
  CUSTOMER_ID NUMBER,
  FIRST_NAME  STRING,
  LAST_NAME   STRING,
  EMAIL       STRING,
  PHONE       STRING,
  ADDRESS     STRING,
  CITY        STRING,
  STATE       STRING,
  ZIP         STRING,
  REGION      STRING,
  UPDATED_AT  TIMESTAMP
);

-- products
CREATE OR REPLACE TABLE RAW.PRODUCTS_RAW (
  PRODUCT_ID   NUMBER,
  PRODUCT_NAME STRING,
  CATEGORY     STRING,
  UNIT_PRICE   NUMBER(10,2),
  UPDATED_AT   TIMESTAMP
);

-- sales (9 columns)
CREATE OR REPLACE TABLE RAW.SALES_RAW (
  TRANSACTION_ID NUMBER,
  CUSTOMER_ID    NUMBER,
  PRODUCT_ID     NUMBER,
  SALE_DATE      DATE,
  QUANTITY       NUMBER,
  UNIT_PRICE     NUMBER(10,2),
  DISCOUNT       FLOAT,
  STORE_ID       STRING,
  UPDATED_AT     TIMESTAMP
);

In [ ]:
USE ROLE ACCOUNTADMIN;
USE DATABASE RETAIL_DW;
USE SCHEMA RAW;

CREATE OR REPLACE FILE FORMAT CSV_FF
  TYPE = CSV
  SKIP_HEADER = 1
  FIELD_OPTIONALLY_ENCLOSED_BY = '"';



In [ ]:
-- Load Customers
COPY INTO RAW.CUSTOMERS_RAW
FROM @RAW.RETAIL_STAGE
PATTERN = '.*customers_2025-08-01\.csv$'
FILE_FORMAT = (FORMAT_NAME = RAW.CSV_FF)
ON_ERROR = CONTINUE;

-- Load Products
COPY INTO RAW.PRODUCTS_RAW
FROM @RAW.RETAIL_STAGE
PATTERN = '.*products_2025-08-01\.csv$'
FILE_FORMAT = (FORMAT_NAME = RAW.CSV_FF)
ON_ERROR = CONTINUE;

-- Load Sales
COPY INTO RAW.SALES_RAW
FROM @RAW.RETAIL_STAGE
PATTERN = '.*sales_2025-08-01\.csv$'
FILE_FORMAT = (FORMAT_NAME = RAW.CSV_FF)
ON_ERROR = CONTINUE;


In [ ]:
SELECT * FROM RAW.sales_raw;

In [ ]:
USE DATABASE RETAIL_DW;
USE SCHEMA DIM;

CREATE OR REPLACE TABLE DIM_CUSTOMER (
  CUSTOMER_SK NUMBER IDENTITY START 1 INCREMENT 1,
  CUSTOMER_ID NUMBER,
  FIRST_NAME STRING,
  LAST_NAME STRING,
  EMAIL STRING,
  PHONE STRING,
  ADDRESS STRING,
  CITY STRING,
  STATE STRING,
  ZIP STRING,
  REGION STRING,
  VALID_FROM TIMESTAMP_NTZ,
  VALID_TO TIMESTAMP_NTZ,
  IS_CURRENT BOOLEAN,
  HASH_DIFF STRING
);

In [ ]:
CREATE OR REPLACE TABLE DIM_PRODUCT (
  PRODUCT_SK NUMBER IDENTITY START 1 INCREMENT 1,
  PRODUCT_ID NUMBER,
  PRODUCT_NAME STRING,
  CATEGORY STRING,
  UNIT_PRICE NUMBER(10,2),
  UPDATED_AT TIMESTAMP_NTZ
);

In [ ]:
USE SCHEMA FACT;

CREATE OR REPLACE TABLE FACT_SALES (
  SALES_SK NUMBER IDENTITY START 1 INCREMENT 1,
  TRANSACTION_ID NUMBER,
  CUSTOMER_SK NUMBER,
  PRODUCT_SK NUMBER,
  SALE_DATE DATE,
  DATE_KEY NUMBER(8),
  QUANTITY NUMBER(10,0),
  UNIT_PRICE NUMBER(10,2),
  DISCOUNT NUMBER(5,2),
  NET_AMOUNT NUMBER(12,2),
  STORE_ID STRING
);

In [ ]:
-- Load Products (SCD1, overwrite)
INSERT INTO DIM.DIM_PRODUCT (PRODUCT_ID, PRODUCT_NAME, CATEGORY, UNIT_PRICE, UPDATED_AT)
SELECT PRODUCT_ID, PRODUCT_NAME, CATEGORY, UNIT_PRICE, UPDATED_AT
FROM RAW.PRODUCTS_RAW;



In [ ]:
-- Load Customers (SCD2 initial)
INSERT INTO DIM.DIM_CUSTOMER
(CUSTOMER_ID,FIRST_NAME,LAST_NAME,EMAIL,PHONE,ADDRESS,CITY,STATE,ZIP,REGION,
 VALID_FROM,VALID_TO,IS_CURRENT,HASH_DIFF)
SELECT
  r.CUSTOMER_ID, r.FIRST_NAME, r.LAST_NAME, r.EMAIL, r.PHONE,
  r.ADDRESS, r.CITY, r.STATE, r.ZIP, r.REGION,
  COALESCE(r.UPDATED_AT, CURRENT_TIMESTAMP()), '9999-12-31'::TIMESTAMP_NTZ, TRUE,
  MD5(TO_JSON(OBJECT_CONSTRUCT_KEEP_NULL(
     'FN',r.FIRST_NAME,'LN',r.LAST_NAME,'EM',r.EMAIL,'PH',r.PHONE,'AD',r.ADDRESS,
     'CT',r.CITY,'ST',r.STATE,'ZP',r.ZIP,'RG',r.REGION)))
FROM RAW.CUSTOMERS_RAW r;

In [ ]:
-- Load Sales Fact
INSERT INTO FACT.FACT_SALES
(TRANSACTION_ID,CUSTOMER_SK,PRODUCT_SK,SALE_DATE,DATE_KEY,QUANTITY,UNIT_PRICE,DISCOUNT,NET_AMOUNT,STORE_ID)
SELECT
  s.TRANSACTION_ID,
  dc.CUSTOMER_SK,
  dp.PRODUCT_SK,
  s.SALE_DATE,
  TO_NUMBER(TO_CHAR(s.SALE_DATE,'YYYYMMDD')),
  s.QUANTITY, s.UNIT_PRICE, s.DISCOUNT,
  ROUND(s.QUANTITY * s.UNIT_PRICE * (1 - COALESCE(s.DISCOUNT,0)), 2),
  s.STORE_ID
FROM RAW.SALES_RAW s
JOIN DIM.DIM_CUSTOMER dc
  ON dc.CUSTOMER_ID = s.CUSTOMER_ID AND dc.IS_CURRENT = TRUE
JOIN DIM.DIM_PRODUCT dp
  ON dp.PRODUCT_ID = s.PRODUCT_ID;

In [ ]:
USE DATABASE RETAIL_DW;
USE SCHEMA RAW;

-- Track changes to customers (for SCD2 updates)
CREATE OR REPLACE STREAM CUSTOMERS_RAW_STRM 
ON TABLE CUSTOMERS_RAW APPEND_ONLY = TRUE;



In [ ]:
-- Track new sales transactions
CREATE OR REPLACE STREAM SALES_RAW_STRM 
ON TABLE SALES_RAW APPEND_ONLY = TRUE;

In [ ]:
USE SCHEMA TRANSFORM;

CREATE OR REPLACE PROCEDURE SP_UPSERT_DIM_CUSTOMER()
RETURNS STRING
LANGUAGE SQL
EXECUTE AS OWNER
AS
$$
BEGIN
  -- Expire old versions if customer attributes changed
  UPDATE DIM.DIM_CUSTOMER d
  SET VALID_TO = s.UPDATED_AT, IS_CURRENT = FALSE
  FROM RAW.CUSTOMERS_RAW_STRM s
  WHERE d.CUSTOMER_ID = s.CUSTOMER_ID
    AND d.IS_CURRENT = TRUE
    AND d.HASH_DIFF <> MD5(TO_JSON(OBJECT_CONSTRUCT_KEEP_NULL(
         'FN',s.FIRST_NAME,'LN',s.LAST_NAME,'EM',s.EMAIL,'PH',s.PHONE,
         'AD',s.ADDRESS,'CT',s.CITY,'ST',s.STATE,'ZP',s.ZIP,'RG',s.REGION)));

  -- Insert new row for changed or new customers
  INSERT INTO DIM.DIM_CUSTOMER
  (CUSTOMER_ID,FIRST_NAME,LAST_NAME,EMAIL,PHONE,ADDRESS,CITY,STATE,ZIP,REGION,
   VALID_FROM,VALID_TO,IS_CURRENT,HASH_DIFF)
  SELECT
    s.CUSTOMER_ID,s.FIRST_NAME,s.LAST_NAME,s.EMAIL,s.PHONE,
    s.ADDRESS,s.CITY,s.STATE,s.ZIP,s.REGION,
    s.UPDATED_AT,'9999-12-31',TRUE,
    MD5(TO_JSON(OBJECT_CONSTRUCT_KEEP_NULL(
       'FN',s.FIRST_NAME,'LN',s.LAST_NAME,'EM',s.EMAIL,'PH',s.PHONE,
       'AD',s.ADDRESS,'CT',s.CITY,'ST',s.STATE,'ZP',s.ZIP,'RG',s.REGION)))
  FROM RAW.CUSTOMERS_RAW_STRM s
  LEFT JOIN DIM.DIM_CUSTOMER d
    ON d.CUSTOMER_ID = s.CUSTOMER_ID AND d.IS_CURRENT = TRUE
       AND d.HASH_DIFF = MD5(TO_JSON(OBJECT_CONSTRUCT_KEEP_NULL(
          'FN',s.FIRST_NAME,'LN',s.LAST_NAME,'EM',s.EMAIL,'PH',s.PHONE,
          'AD',s.ADDRESS,'CT',s.CITY,'ST',s.STATE,'ZP',s.ZIP,'RG',s.REGION)))
  WHERE d.CUSTOMER_ID IS NULL;

  RETURN 'Customer SCD2 Upsert Done';
END;
$$;

In [ ]:
CREATE OR REPLACE PROCEDURE SP_UPSERT_FACT_SALES()
RETURNS STRING
LANGUAGE SQL
EXECUTE AS OWNER
AS
$$
BEGIN
  MERGE INTO FACT.FACT_SALES f
  USING (
    SELECT
      s.TRANSACTION_ID,
      dc.CUSTOMER_SK,
      dp.PRODUCT_SK,
      s.SALE_DATE,
      TO_NUMBER(TO_CHAR(s.SALE_DATE,'YYYYMMDD')) AS DATE_KEY,
      s.QUANTITY, s.UNIT_PRICE, s.DISCOUNT,
      ROUND(s.QUANTITY * s.UNIT_PRICE * (1 - COALESCE(s.DISCOUNT,0)), 2) AS NET_AMOUNT,
      s.STORE_ID
    FROM RAW.SALES_RAW_STRM s
    JOIN DIM.DIM_CUSTOMER dc
      ON dc.CUSTOMER_ID = s.CUSTOMER_ID AND dc.IS_CURRENT = TRUE
    JOIN DIM.DIM_PRODUCT dp
      ON dp.PRODUCT_ID = s.PRODUCT_ID
  ) src
  ON f.TRANSACTION_ID = src.TRANSACTION_ID
  WHEN MATCHED THEN UPDATE SET
    f.CUSTOMER_SK = src.CUSTOMER_SK,
    f.PRODUCT_SK  = src.PRODUCT_SK,
    f.SALE_DATE   = src.SALE_DATE,
    f.DATE_KEY    = src.DATE_KEY,
    f.QUANTITY    = src.QUANTITY,
    f.UNIT_PRICE  = src.UNIT_PRICE,
    f.DISCOUNT    = src.DISCOUNT,
    f.NET_AMOUNT  = src.NET_AMOUNT,
    f.STORE_ID    = src.STORE_ID
  WHEN NOT MATCHED THEN
    INSERT (TRANSACTION_ID,CUSTOMER_SK,PRODUCT_SK,SALE_DATE,DATE_KEY,QUANTITY,UNIT_PRICE,DISCOUNT,NET_AMOUNT,STORE_ID)
    VALUES (src.TRANSACTION_ID,src.CUSTOMER_SK,src.PRODUCT_SK,src.SALE_DATE,src.DATE_KEY,src.QUANTITY,src.UNIT_PRICE,src.DISCOUNT,src.NET_AMOUNT,src.STORE_ID);

  RETURN 'Fact Sales Upsert Done';
END;
$$;

In [ ]:
-- Load Customers
COPY INTO RAW.CUSTOMERS_RAW
FROM @RAW.RETAIL_STAGE
PATTERN = '.*customers_2025-08-02\.csv$'
FILE_FORMAT = (FORMAT_NAME = RAW.CSV_FF)
ON_ERROR = CONTINUE;

-- Load Products
COPY INTO RAW.SALES_RAW
FROM @RAW.RETAIL_STAGE
PATTERN = '.*products_2025-08-02\.csv$'
FILE_FORMAT = (FORMAT_NAME = RAW.CSV_FF)
ON_ERROR = CONTINUE;


In [ ]:
CREATE OR REPLACE STREAM RAW.SALES_RAW_STRM ON TABLE RAW.SALES_RAW;

CREATE OR REPLACE TASK FACT.LOAD_FACT_SALES_TASK
  WAREHOUSE = COMPUTE_WH
  SCHEDULE = '1 MINUTE'
  WHEN SYSTEM$STREAM_HAS_DATA('RAW.SALES_RAW_STRM')
AS
INSERT INTO FACT.FACT_SALES (
  TRANSACTION_ID, CUSTOMER_ID, PRODUCT_ID, SALE_DATE,
  QUANTITY, UNIT_PRICE, DISCOUNT, STORE_ID, UPDATED_AT
)
SELECT TRANSACTION_ID, CUSTOMER_ID, PRODUCT_ID, SALE_DATE,
       QUANTITY, UNIT_PRICE, DISCOUNT, STORE_ID, UPDATED_AT
FROM RAW.SALES_RAW_STRM;

ALTER TASK FACT.LOAD_FACT_SALES_TASK RESUME;

In [ ]:
COPY INTO RAW.CUSTOMERS_RAW
FROM @RAW.RETAIL_STAGE
FILE_FORMAT = (TYPE = CSV FIELD_OPTIONALLY_ENCLOSED_BY='"' SKIP_HEADER=1)
PATTERN = '.customers.\.csv';

COPY INTO RAW.PRODUCTS_RAW
FROM @RAW.RETAIL_STAGE
FILE_FORMAT = (TYPE = CSV FIELD_OPTIONALLY_ENCLOSED_BY='"' SKIP_HEADER=1)
PATTERN = '.products.\.csv';

COPY INTO RAW.SALES_RAW
FROM @RAW.RETAIL_STAGE
FILE_FORMAT = (TYPE = CSV FIELD_OPTIONALLY_ENCLOSED_BY='"' SKIP_HEADER=1)
PATTERN = '.sales.\.csv';

In [ ]:
-- Create a sequence (if not already)
CREATE OR REPLACE SEQUENCE DIM.PRODUCT_SK_SEQ START = 1 INCREMENT = 1;

-- Load products
INSERT INTO DIM.DIM_PRODUCT (
  PRODUCT_SK, PRODUCT_ID, PRODUCT_NAME, CATEGORY, UNIT_PRICE, UPDATED_AT
)
SELECT 
  DIM.PRODUCT_SK_SEQ.NEXTVAL,   -- auto-generate surrogate key
  PRODUCT_ID,
  PRODUCT_NAME,
  CATEGORY,
  UNIT_PRICE,
  UPDATED_AT
FROM RAW.PRODUCTS_RAW;

In [ ]:
CREATE OR REPLACE SEQUENCE DIM.CUSTOMER_SK_SEQ START = 1 INCREMENT = 1;

INSERT INTO DIM.DIM_CUSTOMER (
  CUSTOMER_ID, FIRST_NAME, LAST_NAME, EMAIL, PHONE, ADDRESS, CITY, STATE, ZIP, REGION,
  START_DATE, END_DATE, IS_CURRENT
)
SELECT 
  CUSTOMER_ID, FIRST_NAME, LAST_NAME, EMAIL, PHONE, ADDRESS, CITY, STATE, ZIP, REGION,
  CURRENT_DATE, NULL, TRUE
FROM RAW.CUSTOMERS_RAW;

In [ ]:
CREATE OR REPLACE TABLE DIM.DIM_CUSTOMER (
  CUSTOMER_SK NUMBER DEFAULT DIM.CUSTOMER_SK_SEQ.NEXTVAL,  -- surrogate key
  CUSTOMER_ID NUMBER,
  FIRST_NAME  STRING,
  LAST_NAME   STRING,
  EMAIL       STRING,
  PHONE       STRING,
  ADDRESS     STRING,
  CITY        STRING,
  STATE       STRING,
  ZIP         STRING,
  REGION      STRING,
  START_DATE  DATE,
  END_DATE    DATE,
  IS_CURRENT  BOOLEAN
);

In [ ]:
CREATE OR REPLACE SEQUENCE DIM.CUSTOMER_SK_SEQ START = 1 INCREMENT = 1;

INSERT INTO DIM.DIM_CUSTOMER (
  CUSTOMER_ID, FIRST_NAME, LAST_NAME, EMAIL, PHONE, ADDRESS, CITY, STATE, ZIP, REGION,
  START_DATE, END_DATE, IS_CURRENT
)
SELECT 
  CUSTOMER_ID, FIRST_NAME, LAST_NAME, EMAIL, PHONE, ADDRESS, CITY, STATE, ZIP, REGION,
  CURRENT_DATE, NULL, TRUE
FROM RAW.CUSTOMERS_RAW;

In [ ]:
CREATE OR REPLACE SEQUENCE DIM.CUSTOMER_SK_SEQ START = 1 INCREMENT = 1;

In [ ]:
CREATE OR REPLACE TABLE DIM.DIM_CUSTOMER (
  CUSTOMER_SK NUMBER DEFAULT DIM.CUSTOMER_SK_SEQ.NEXTVAL,
  CUSTOMER_ID NUMBER,
  FIRST_NAME  STRING,
  LAST_NAME   STRING,
  EMAIL       STRING,
  PHONE       STRING,
  ADDRESS     STRING,
  CITY        STRING,
  STATE       STRING,
  ZIP         STRING,
  REGION      STRING,
  START_DATE  DATE,
  END_DATE    DATE,
  IS_CURRENT  BOOLEAN
);

In [ ]:
INSERT INTO DIM.DIM_CUSTOMER (
  CUSTOMER_ID, FIRST_NAME, LAST_NAME, EMAIL, PHONE, ADDRESS, CITY, STATE, ZIP, REGION,
  START_DATE, END_DATE, IS_CURRENT
)
SELECT
  CUSTOMER_ID, FIRST_NAME, LAST_NAME, EMAIL, PHONE, ADDRESS, CITY, STATE, ZIP, REGION,
  CURRENT_DATE, NULL, TRUE
FROM RAW.CUSTOMERS_RAW;

In [ ]:
MERGE INTO DIM.DIM_CUSTOMER tgt
USING RAW.CUSTOMERS_RAW src
ON tgt.CUSTOMER_ID = src.CUSTOMER_ID
   AND tgt.IS_CURRENT = TRUE

WHEN MATCHED AND (
     NVL(tgt.FIRST_NAME,'') <> NVL(src.FIRST_NAME,'')
  OR NVL(tgt.LAST_NAME,'')  <> NVL(src.LAST_NAME,'')
  OR NVL(tgt.EMAIL,'')      <> NVL(src.EMAIL,'')
  OR NVL(tgt.PHONE,'')      <> NVL(src.PHONE,'')
  OR NVL(tgt.ADDRESS,'')    <> NVL(src.ADDRESS,'')
  OR NVL(tgt.CITY,'')       <> NVL(src.CITY,'')
  OR NVL(tgt.STATE,'')      <> NVL(src.STATE,'')
  OR NVL(tgt.ZIP,'')        <> NVL(src.ZIP,'')
  OR NVL(tgt.REGION,'')     <> NVL(src.REGION,'')
)
THEN UPDATE SET
  tgt.END_DATE   = CURRENT_DATE,
  tgt.IS_CURRENT = FALSE

WHEN NOT MATCHED THEN
INSERT (
  CUSTOMER_SK, CUSTOMER_ID, FIRST_NAME, LAST_NAME, EMAIL, PHONE, ADDRESS,
  CITY, STATE, ZIP, REGION, START_DATE, END_DATE, IS_CURRENT
)
VALUES (
  DIM.CUSTOMER_SK_SEQ.NEXTVAL,
  src.CUSTOMER_ID, src.FIRST_NAME, src.LAST_NAME, src.EMAIL, src.PHONE, src.ADDRESS,
  src.CITY, src.STATE, src.ZIP, src.REGION,
  CURRENT_DATE, NULL, TRUE
);

In [ ]:
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE COMPUTE_WH;

-- Make sure you’re in the right DB & schema
USE DATABASE RETAIL_DW;
USE SCHEMA RAW;

INSERT INTO RAW.CUSTOMERS_RAW VALUES 
  (101, 'John', 'Doe', 'john.doe.new@email.com', '555-1234',
   '123 Main St', 'Seattle', 'WA', '98101', 'West', CURRENT_TIMESTAMP);


In [ ]:
MERGE INTO DIM.DIM_CUSTOMER tgt
USING RAW.CUSTOMERS_RAW src
ON tgt.CUSTOMER_ID = src.CUSTOMER_ID
   AND tgt.IS_CURRENT = TRUE

WHEN MATCHED AND (
     NVL(tgt.FIRST_NAME,'') <> NVL(src.FIRST_NAME,'')
  OR NVL(tgt.LAST_NAME,'')  <> NVL(src.LAST_NAME,'')
  OR NVL(tgt.EMAIL,'')      <> NVL(src.EMAIL,'')
  OR NVL(tgt.PHONE,'')      <> NVL(src.PHONE,'')
  OR NVL(tgt.ADDRESS,'')    <> NVL(src.ADDRESS,'')
  OR NVL(tgt.CITY,'')       <> NVL(src.CITY,'')
  OR NVL(tgt.STATE,'')      <> NVL(src.STATE,'')
  OR NVL(tgt.ZIP,'')        <> NVL(src.ZIP,'')
  OR NVL(tgt.REGION,'')     <> NVL(src.REGION,'')
)
THEN UPDATE SET
  tgt.END_DATE   = CURRENT_DATE,
  tgt.IS_CURRENT = FALSE

WHEN NOT MATCHED THEN
INSERT (
  CUSTOMER_SK, CUSTOMER_ID, FIRST_NAME, LAST_NAME, EMAIL, PHONE, ADDRESS,
  CITY, STATE, ZIP, REGION, START_DATE, END_DATE, IS_CURRENT
)
VALUES (
  DIM.CUSTOMER_SK_SEQ.NEXTVAL,
  src.CUSTOMER_ID, src.FIRST_NAME, src.LAST_NAME, src.EMAIL, src.PHONE, src.ADDRESS,
  src.CITY, src.STATE, src.ZIP, src.REGION,
  CURRENT_DATE, NULL, TRUE
);

In [ ]:
SELECT * 
FROM RETAIL_DW.DIM.DIM_CUSTOMER
WHERE CUSTOMER_ID = 101
ORDER BY START_DATE DESC;

In [ ]:
DESC TABLE RETAIL_DW.DIM.DIM_CUSTOMER;

In [ ]:
SHOW SEQUENCES IN SCHEMA RETAIL_DW.DIM;

In [ ]:
MERGE INTO RETAIL_DW.DIM.DIM_CUSTOMER tgt
USING RETAIL_DW.RAW.CUSTOMERS_RAW src
ON tgt.CUSTOMER_ID = src.CUSTOMER_ID
   AND tgt.IS_CURRENT = TRUE

WHEN MATCHED AND (
     NVL(tgt.FIRST_NAME,'') <> NVL(src.FIRST_NAME,'')
  OR NVL(tgt.LAST_NAME,'')  <> NVL(src.LAST_NAME,'')
  OR NVL(tgt.EMAIL,'')      <> NVL(src.EMAIL,'')
  OR NVL(tgt.PHONE,'')      <> NVL(src.PHONE,'')
  OR NVL(tgt.ADDRESS,'')    <> NVL(src.ADDRESS,'')
  OR NVL(tgt.CITY,'')       <> NVL(src.CITY,'')
  OR NVL(tgt.STATE,'')      <> NVL(src.STATE,'')
  OR NVL(tgt.ZIP,'')        <> NVL(src.ZIP,'')
  OR NVL(tgt.REGION,'')     <> NVL(src.REGION,'')
)
THEN UPDATE SET
  tgt.END_DATE   = CURRENT_DATE,
  tgt.IS_CURRENT = FALSE

WHEN NOT MATCHED THEN
INSERT (
  CUSTOMER_SK, CUSTOMER_ID, FIRST_NAME, LAST_NAME, EMAIL, PHONE, ADDRESS,
  CITY, STATE, ZIP, REGION, START_DATE, END_DATE, IS_CURRENT
)
VALUES (
  RETAIL_DW.DIM.CUSTOMER_SK_SEQ.NEXTVAL,
  src.CUSTOMER_ID, src.FIRST_NAME, src.LAST_NAME, src.EMAIL, src.PHONE, src.ADDRESS,
  src.CITY, src.STATE, src.ZIP, src.REGION,
  CURRENT_DATE, NULL, TRUE
);

In [ ]:
INSERT INTO RETAIL_DW.RAW.CUSTOMERS_RAW VALUES 
  (101, 'John', 'Doe', 'john.new@email.com', '555-4321',
   '123 Main St', 'Seattle', 'WA', '98101', 'West', CURRENT_TIMESTAMP);

In [ ]:
SELECT CUSTOMER_ID, EMAIL, PHONE, START_DATE, END_DATE, IS_CURRENT
FROM RETAIL_DW.DIM.DIM_CUSTOMER
WHERE CUSTOMER_ID = 101
ORDER BY START_DATE DESC;

In [ ]:
-- Step 1: Close out current records if data changed
UPDATE RETAIL_DW.DIM.DIM_CUSTOMER tgt
SET END_DATE = CURRENT_DATE,
    IS_CURRENT = FALSE
FROM RETAIL_DW.RAW.CUSTOMERS_RAW src
WHERE tgt.CUSTOMER_ID = src.CUSTOMER_ID
  AND tgt.IS_CURRENT = TRUE
  AND (
       NVL(tgt.FIRST_NAME,'') <> NVL(src.FIRST_NAME,'')
    OR NVL(tgt.LAST_NAME,'')  <> NVL(src.LAST_NAME,'')
    OR NVL(tgt.EMAIL,'')      <> NVL(src.EMAIL,'')
    OR NVL(tgt.PHONE,'')      <> NVL(src.PHONE,'')
    OR NVL(tgt.ADDRESS,'')    <> NVL(src.ADDRESS,'')
    OR NVL(tgt.CITY,'')       <> NVL(src.CITY,'')
    OR NVL(tgt.STATE,'')      <> NVL(src.STATE,'')
    OR NVL(tgt.ZIP,'')        <> NVL(src.ZIP,'')
    OR NVL(tgt.REGION,'')     <> NVL(src.REGION,'')
  );

-- Step 2: Insert a new current record if not already present
INSERT INTO RETAIL_DW.DIM.DIM_CUSTOMER (
  CUSTOMER_SK, CUSTOMER_ID, FIRST_NAME, LAST_NAME, EMAIL, PHONE, ADDRESS,
  CITY, STATE, ZIP, REGION, START_DATE, END_DATE, IS_CURRENT
)
SELECT
  RETAIL_DW.DIM.CUSTOMER_SK_SEQ.NEXTVAL,
  src.CUSTOMER_ID, src.FIRST_NAME, src.LAST_NAME, src.EMAIL, src.PHONE, src.ADDRESS,
  src.CITY, src.STATE, src.ZIP, src.REGION,
  CURRENT_DATE, NULL, TRUE
FROM RETAIL_DW.RAW.CUSTOMERS_RAW src
LEFT JOIN RETAIL_DW.DIM.DIM_CUSTOMER tgt
  ON src.CUSTOMER_ID = tgt.CUSTOMER_ID
 AND tgt.IS_CURRENT = TRUE
WHERE tgt.CUSTOMER_ID IS NULL;

In [ ]:
SELECT CUSTOMER_ID, EMAIL, PHONE, START_DATE, END_DATE, IS_CURRENT
FROM RETAIL_DW.DIM.DIM_CUSTOMER
WHERE CUSTOMER_ID = 101
ORDER BY START_DATE DESC;

In [ ]:
CREATE OR REPLACE TASK DIM.CUSTOMER_SCD2_TASK
  WAREHOUSE = COMPUTE_WH
  SCHEDULE = 'USING CRON 0 2 * * * UTC'  -- runs daily at 2AM UTC
AS
-- Step 1: Close out current rows if data changed
BEGIN
    UPDATE RETAIL_DW.DIM.DIM_CUSTOMER tgt
    SET END_DATE = CURRENT_DATE,
        IS_CURRENT = FALSE
    FROM RETAIL_DW.RAW.CUSTOMERS_RAW src
    WHERE tgt.CUSTOMER_ID = src.CUSTOMER_ID
      AND tgt.IS_CURRENT = TRUE
      AND (
           NVL(tgt.FIRST_NAME,'') <> NVL(src.FIRST_NAME,'')
        OR NVL(tgt.LAST_NAME,'')  <> NVL(src.LAST_NAME,'')
        OR NVL(tgt.EMAIL,'')      <> NVL(src.EMAIL,'')
        OR NVL(tgt.PHONE,'')      <> NVL(src.PHONE,'')
        OR NVL(tgt.ADDRESS,'')    <> NVL(src.ADDRESS,'')
        OR NVL(tgt.CITY,'')       <> NVL(src.CITY,'')
        OR NVL(tgt.STATE,'')      <> NVL(src.STATE,'')
        OR NVL(tgt.ZIP,'')        <> NVL(src.ZIP,'')
        OR NVL(tgt.REGION,'')     <> NVL(src.REGION,'')
      );

    -- Step 2: Insert new current rows
    INSERT INTO RETAIL_DW.DIM.DIM_CUSTOMER (
      CUSTOMER_SK, CUSTOMER_ID, FIRST_NAME, LAST_NAME, EMAIL, PHONE, ADDRESS,
      CITY, STATE, ZIP, REGION, START_DATE, END_DATE, IS_CURRENT
    )
    SELECT
      RETAIL_DW.DIM.CUSTOMER_SK_SEQ.NEXTVAL,
      src.CUSTOMER_ID, src.FIRST_NAME, src.LAST_NAME, src.EMAIL, src.PHONE, src.ADDRESS,
      src.CITY, src.STATE, src.ZIP, src.REGION,
      CURRENT_DATE, NULL, TRUE
    FROM RETAIL_DW.RAW.CUSTOMERS_RAW src
    LEFT JOIN RETAIL_DW.DIM.DIM_CUSTOMER tgt
      ON src.CUSTOMER_ID = tgt.CUSTOMER_ID
     AND tgt.IS_CURRENT = TRUE
    WHERE tgt.CUSTOMER_ID IS NULL;
END;

In [ ]:
ALTER TASK DIM.CUSTOMER_SCD2_TASK RESUME;

In [ ]:
EXECUTE TASK DIM.CUSTOMER_SCD2_TASK;

In [ ]:
EXECUTE TASK DIM.CUSTOMER_SCD2_TASK;

In [ ]:
SELECT CUSTOMER_ID, EMAIL, PHONE, START_DATE, END_DATE, IS_CURRENT
FROM RETAIL_DW.DIM.DIM_CUSTOMER
WHERE CUSTOMER_ID = 101
ORDER BY START_DATE DESC;